# YOLOv8 Training Loop & Failure Analysis
This notebook isolates the YOLOv8 training loop based on your exact code, grid search evaluations, and qualitative NMS failure analysis grid.

In [ ]:
!pip install ultralytics -q
from ultralytics import YOLO
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import cv2


In [ ]:
# YOLOv8 Training (Exact params from Untitled0)
model = YOLO("yolov8n.pt")
results = model.train(
    data="./Military-vehicles-detection-1/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    val=True,
    project="military_yolo_research",
    name="yolov8n_baseline",
    seed=42,
)


In [ ]:
# 1. Plotting training metrics
df_results = pd.read_csv('/content/runs/detect/military_yolo_research/yolov8n_baseline/results.csv')
sns.set_theme(style="whitegrid")

plt.figure(figsize=(10, 6))
sns.lineplot(data=df_results, x='epoch', y='metrics/mAP50(B)',label='mAP50(B)')
sns.lineplot(data=df_results, x='epoch', y='metrics/mAP50-95(B)', label='mAP50-95(B)')
plt.xlabel('Epoch')
plt.ylabel('Metrics')
plt.legend()
plt.show()


## Confidence and IoU Threshold Grid Search


In [ ]:
best_model = YOLO("/content/runs/detect/military_yolo_research/yolov8n_baseline/weights/best.pt")
data_yaml_path = "./Military-vehicles-detection-1/data.yaml"

conf_thresholds = np.arange(0.01, 0.71, 0.05)
precisions, recalls, map50s = [], [], []

for conf in conf_thresholds:
    metrics = best_model.val(data=data_yaml_path, conf=conf, iou=0.60, verbose=False, split='test')
    precisions.append(metrics.results_dict["metrics/precision(B)"])
    recalls.append(metrics.results_dict["metrics/recall(B)"])
    map50s.append(metrics.results_dict["metrics/mAP50(B)"])

plt.figure(figsize=(12, 7))
plt.plot(conf_thresholds, precisions, label="Precision", marker="o")
plt.plot(conf_thresholds, recalls, label="Recall", marker="s")
plt.title("Metrics vs Confidence Threshold")
plt.legend()
plt.show()


## Qualitative Failure Analysis Grid (GT vs Predictions)


In [ ]:
import glob
import random
val_images_dir = "./Military-vehicles-detection-1/test/images"
val_labels_dir = "./Military-vehicles-detection-1/test/labels"
class_names = best_model.names

image_paths = glob.glob(os.path.join(val_images_dir, "*.jpg"))
random.seed(12)
selected_images = random.sample(image_paths, 3)

for img_path in selected_images:
    # Inference and side-by-side plotting logic from original code
    img = cv2.imread(img_path)
    # (In actual notebook, the full cv2 bounding box mapping logic is here)
    results = best_model.predict(img_path, conf=0.25, iou=0.50, verbose=False)[0]
    # ...
